# PDF Extraction Experiment

**Goal:** Find the best extraction + chunking combination for production PDF ingestion.

**Experiment structure (3 phases, each isolates one variable):**

| Phase | Variable | Fixed | Output |
|---|---|---|---|
| 1 | Extraction method | — | Visual inspection |
| 2 | Chunking strategy | Phase 1 winner extraction | RAGAS scores |
| 3 | Contextual enrichment | Phase 2 winner chunking | RAGAS scores |

**PDFs under test:** 5 real-world documents (financial reports, academic paper, API docs)


In [16]:
import sys, os, re, json, asyncio, unicodedata, warnings
from pathlib import Path
from dataclasses import dataclass, field
from collections import Counter
from typing import Literal

import numpy as np
import tiktoken
import pymupdf
import pdfplumber

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

warnings.filterwarnings("ignore")

PDF_DIR = ROOT / "data" / "raw" / "pdfs"
EVAL_DIR = ROOT / "eval" / "golden_dataset" / "pdf"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

PDFS = sorted(PDF_DIR.glob("*.pdf"))
MAX_PAGES = 15  # limit per PDF to keep experiment fast

print(f"Found {len(PDFS)} PDFs:")
for p in PDFS:
    doc = pymupdf.open(str(p))
    print(f"  {p.name}  —  {len(doc)} pages")
    doc.close()


Found 5 PDFs:
  1910.10683v4.pdf  —  67 pages
  Annual-Report-2023.pdf  —  104 pages
  FY23_Q4_Consolidated_Financial_Statements.pdf  —  3 pages
  P180107173682d0431bf651fded74199f10.pdf  —  177 pages
  _10-K-Q4-2023-As-Filed.pdf  —  80 pages


## Phase 1: Extraction Strategy Comparison

**Strategy A (baseline):** `get_text()` — raw dump, no layout awareness  
**Strategy B (candidate):** layout-aware reading order + pdfplumber table detection + cleaning pipeline

**Cleaning pipeline (fixed, applied to both):**
- Hyphenation repair: `infor-\nmation` → `information`
- Ligature normalization: `ﬁnancial` → `financial`  
- Header/footer removal: lines appearing on ≥70% of pages stripped
- Quality gate: pages with <50 chars flagged as likely scanned images


In [17]:
def fix_hyphenation(text: str) -> str:
    return re.sub(r'(\w+)-\n(\w+)', r'\1\2', text)

def normalize_ligatures(text: str) -> str:
    return unicodedata.normalize("NFKD", text)

def remove_headers_footers(pages: list[str]) -> list[str]:
    """Strip lines that repeat on ≥70% of pages (page numbers, doc titles, footers)."""
    all_lines = []
    for page in pages:
        all_lines.extend(l.strip() for l in page.splitlines() if l.strip())
    counts = Counter(all_lines)
    threshold = max(2, len(pages) * 0.7)
    noise = {line for line, cnt in counts.items() if cnt >= threshold}

    cleaned = []
    for page in pages:
        lines = [l for l in page.splitlines() if l.strip() not in noise]
        cleaned.append("\n".join(lines))
    return cleaned

def quality_gate(pages: list[str]) -> list[dict]:
    """Flag pages with suspiciously low text content."""
    report = []
    for i, page in enumerate(pages):
        char_count = len(page.strip())
        report.append({
            "page": i + 1,
            "chars": char_count,
            "quality": "ok" if char_count >= 50 else "LOW — likely scanned image or blank"
        })
    return report

def clean_pipeline(pages: list[str]) -> list[str]:
    pages = [fix_hyphenation(p) for p in pages]
    pages = [normalize_ligatures(p) for p in pages]
    pages = remove_headers_footers(pages)
    return pages


In [18]:
def extract_strategy_a(pdf_path: Path, max_pages: int = MAX_PAGES) -> str:
    """Baseline: raw get_text(), no layout awareness, cleaning applied."""
    doc = pymupdf.open(str(pdf_path))
    pages = []
    for i, page in enumerate(doc):
        if i >= max_pages:
            break
        pages.append(page.get_text())
    doc.close()
    pages = clean_pipeline(pages)
    return "\n\n---PAGE BREAK---\n\n".join(pages)


In [19]:
@dataclass
class ExtractedBlock:
    type: Literal["heading", "paragraph", "table", "code", "caption"]
    content: str
    page: int
    section: str = ""
    heading_level: int = 0  # 1=H1, 2=H2, 3=H3; 0 if not heading


def _get_body_font_size(page_dict: dict) -> float:
    sizes = []
    for block in page_dict.get("blocks", []):
        if block.get("type") != 0:
            continue
        for line in block.get("lines", []):
            for span in line.get("spans", []):
                sizes.append(round(span["size"], 1))
    if not sizes:
        return 11.0
    return Counter(sizes).most_common(1)[0][0]


def _is_monospace(font_name: str) -> bool:
    return any(kw in font_name.lower() for kw in
               ("mono", "courier", "code", "consol", "inconsolata", "fixed", "typewriter"))


def _extract_text_blocks(page, page_num: int, table_bboxes: list) -> list[ExtractedBlock]:
    page_dict = page.get_text("dict")
    body_size = _get_body_font_size(page_dict)
    blocks_out: list[ExtractedBlock] = []

    raw_blocks = [b for b in page_dict["blocks"] if b.get("type") == 0]
    raw_blocks.sort(key=lambda b: (round(b["bbox"][1] / 20) * 20, b["bbox"][0]))

    def _overlaps(b1, b2) -> bool:
        return not (b1[2] < b2[0] or b1[0] > b2[2] or b1[3] < b2[1] or b1[1] > b2[3])

    for block in raw_blocks:
        if any(_overlaps(block["bbox"], tb) for tb in table_bboxes):
            continue

        lines_text = []
        is_mono_block = False
        max_size = 0.0
        is_bold = False

        for line in block.get("lines", []):
            span_texts = []
            for span in line.get("spans", []):
                t = span["text"].strip()
                if not t:
                    continue
                span_texts.append(t)
                if _is_monospace(span.get("font", "")):
                    is_mono_block = True
                sz = round(span["size"], 1)
                if sz > max_size:
                    max_size = sz
                if "bold" in span.get("font", "").lower():
                    is_bold = True
            if span_texts:
                lines_text.append(" ".join(span_texts))

        content = "\n".join(lines_text).strip()
        if not content:
            continue

        if is_mono_block:
            blocks_out.append(ExtractedBlock("code", content, page_num))
        elif max_size >= body_size + 5 or (max_size >= body_size + 3 and is_bold):
            blocks_out.append(ExtractedBlock("heading", content, page_num, heading_level=1))
        elif max_size >= body_size + 2.5 or (max_size >= body_size + 1 and is_bold):
            blocks_out.append(ExtractedBlock("heading", content, page_num, heading_level=2))
        elif max_size >= body_size + 0.5 and is_bold:
            blocks_out.append(ExtractedBlock("heading", content, page_num, heading_level=3))
        else:
            blocks_out.append(ExtractedBlock("paragraph", content, page_num))

    return blocks_out


def _extract_tables_pdfplumber(pdf_path: Path, page_num: int) -> list[ExtractedBlock]:
    """Use pdfplumber for accurate table extraction on a single page."""
    tables_out = []
    try:
        with pdfplumber.open(str(pdf_path)) as pdf:
            if page_num >= len(pdf.pages):
                return []
            page = pdf.pages[page_num]
            for table in page.extract_tables():
                if not table or len(table) < 2 or len(table[0]) < 2:
                    continue
                headers = [str(h or "").strip() for h in table[0]]
                rows = [[str(c or "").strip() for c in row] for row in table[1:]]
                header_row = "| " + " | ".join(headers) + " |"
                sep = "| " + " | ".join(["---"] * len(headers)) + " |"
                data_rows = ["| " + " | ".join(row) + " |" for row in rows if any(row)]
                md = "\n".join([header_row, sep] + data_rows)
                tables_out.append(ExtractedBlock("table", md, page_num + 1))
    except Exception as e:
        pass
    return tables_out


def extract_strategy_b(pdf_path: Path, max_pages: int = MAX_PAGES) -> list[ExtractedBlock]:
    """Layout-aware extraction: reading order + pdfplumber tables + heading detection."""
    doc = pymupdf.open(str(pdf_path))
    all_blocks: list[ExtractedBlock] = []
    current_section = ""

    for page_num in range(min(max_pages, len(doc))):
        page = doc[page_num]

        # Get table bboxes from PyMuPDF to exclude from text extraction
        try:
            mupdf_tables = page.find_tables()
            table_bboxes = [t.bbox for t in mupdf_tables.tables]
        except Exception:
            table_bboxes = []

        # Text blocks (headings, paragraphs, code)
        text_blocks = _extract_text_blocks(page, page_num + 1, table_bboxes)
        for b in text_blocks:
            if b.type == "heading":
                current_section = b.content
            b.section = current_section
        all_blocks.extend(text_blocks)

        # Tables via pdfplumber
        table_blocks = _extract_tables_pdfplumber(pdf_path, page_num)
        for b in table_blocks:
            b.section = current_section
        all_blocks.extend(table_blocks)

    doc.close()

    # Clean non-table blocks
    text_pages: dict[int, list[ExtractedBlock]] = {}
    for b in all_blocks:
        if b.type != "table":
            text_pages.setdefault(b.page, []).append(b)

    page_texts = {pg: "\n".join(b.content for b in blocks)
                  for pg, blocks in text_pages.items()}
    pages_list = [page_texts.get(i + 1, "") for i in range(max_pages)]
    cleaned = clean_pipeline(pages_list)
    cleaned_map = {i + 1: cleaned[i] for i in range(len(cleaned))}

    # Rebuild text blocks with cleaned content
    final_blocks: list[ExtractedBlock] = []
    text_block_pages: dict[int, list[ExtractedBlock]] = {}
    table_block_list: list[ExtractedBlock] = []

    for b in all_blocks:
        if b.type == "table":
            table_block_list.append(b)
        else:
            text_block_pages.setdefault(b.page, []).append(b)

    # Re-split cleaned pages back into blocks (approximate)
    for pg, clean_text in cleaned_map.items():
        for b in text_block_pages.get(pg, []):
            # Check if original content survived cleaning
            if b.content.strip() and any(
                word in clean_text for word in b.content.split()[:3] if len(word) > 3
            ):
                final_blocks.append(b)

    final_blocks.extend(table_block_list)
    final_blocks.sort(key=lambda b: (b.page, b.type != "heading"))
    return final_blocks


def blocks_to_text(blocks: list[ExtractedBlock]) -> str:
    """Flatten blocks to plain text (for SlidingWindow chunker input)."""
    parts = []
    for b in blocks:
        if b.type == "heading":
            prefix = "#" * b.heading_level if b.heading_level else "##"
            parts.append(f"{prefix} {b.content}")
        elif b.type == "table":
            parts.append(b.content)
        elif b.type == "code":
            parts.append(f"```\n{b.content}\n```")
        else:
            parts.append(b.content)
    return "\n\n".join(parts)


In [22]:
results_a = {}
results_b = {}

for pdf in PDFS:
    print(f"\nProcessing: {pdf.name}")
    results_a[pdf.name] = extract_strategy_a(pdf)
    results_b[pdf.name] = extract_strategy_b(pdf)
    blocks = results_b[pdf.name]
    headings = [b for b in blocks if b.type == "heading"]
    tables = [b for b in blocks if b.type == "table"]
    code = [b for b in blocks if b.type == "code"]
    print(f"  Strategy A → {len(results_a[pdf.name].split())} words")
    print(f"  Strategy B → {len(blocks)} blocks | {len(headings)} headings | {len(tables)} tables | {len(code)} code blocks")



Processing: 1910.10683v4.pdf
  Strategy A → 7807 words
  Strategy B → 129 blocks | 15 headings | 1 tables | 23 code blocks

Processing: Annual-Report-2023.pdf
  Strategy A → 3182 words
  Strategy B → 31 blocks | 7 headings | 5 tables | 0 code blocks

Processing: FY23_Q4_Consolidated_Financial_Statements.pdf
  Strategy A → 697 words
  Strategy B → 26 blocks | 0 headings | 3 tables | 0 code blocks

Processing: P180107173682d0431bf651fded74199f10.pdf
  Strategy A → 8642 words
  Strategy B → 187 blocks | 58 headings | 0 tables | 0 code blocks

Processing: _10-K-Q4-2023-As-Filed.pdf
  Strategy A → 10188 words
  Strategy B → 212 blocks | 7 headings | 1 tables | 0 code blocks


In [23]:
TARGET_PDF = PDFS[0].name  # change index to inspect different PDFs

print(f"PDF: {TARGET_PDF}")
print("\n" + "="*60)
print("STRATEGY A — first 1200 chars")
print("="*60)
print(results_a[TARGET_PDF][:1200])

print("\n" + "="*60)
print("STRATEGY B — first 1200 chars (flattened blocks)")
print("="*60)
b_text = blocks_to_text(results_b[TARGET_PDF])
print(b_text[:1200])


PDF: 1910.10683v4.pdf

STRATEGY A — first 1200 chars
Journal of Machine Learning Research 21 (2020) 1-67
Submitted 1/20; Revised 6/20; Published 6/20
Exploring the Limits of Transfer Learning with a Unified
Text-to-Text Transformer
Colin Raffel∗
craffel@gmail.com
Noam Shazeer∗
noam@google.com
Adam Roberts∗
adarob@google.com
Katherine Lee∗
katherinelee@google.com
Sharan Narang
sharannarang@google.com
Michael Matena
mmatena@google.com
Yanqi Zhou
yanqiz@google.com
Wei Li
mweili@google.com
Peter J. Liu
peterjliu@google.com
Google, Mountain View, CA 94043, USA
Editor: Ivan Titov
Abstract
Transfer learning, where a model is first pre-trained on a data-rich task before being finetuned on a downstream task, has emerged as a powerful technique in natural language
processing (NLP). The effectiveness of transfer learning has given rise to a diversity of
approaches, methodology, and practice. In this paper, we explore the landscape of transfer
learning techniques for NLP by introducing a unified f

In [24]:
TARGET_PDF = PDFS[1].name

print(f"PDF: {TARGET_PDF}")
print("\n" + "="*60)
print("STRATEGY A — first 1200 chars")
print("="*60)
print(results_a[TARGET_PDF][:1200])

print("\n" + "="*60)
print("STRATEGY B — first 1200 chars (flattened blocks)")
print("="*60)
b_text = blocks_to_text(results_b[TARGET_PDF])
print(b_text[:1200])


PDF: Annual-Report-2023.pdf

STRATEGY A — first 1200 chars
AFRICA
GROUP I CONSTITUENCY
REPORT
20
23
Dr. Floribert Ngaruko
Executive Director, 
AFRICA GROUP I CONSTITUENCY
ANNUAL

---PAGE BREAK---



---PAGE BREAK---

AFRICA
GROUP I CONSTITUENCY
REPORT
20
23
Dr. Floribert Ngaruko
Executive Director, 
AFRICA GROUP I CONSTITUENCY
ANNUAL

---PAGE BREAK---



---PAGE BREAK---

Contents
Acronyms and Abbreviations............................................................................................................... v
Selected Development Indicators for Constituency Countries..................................................... vi
Foreward................................................................................................................................................. 1
Executive Summary.............................................................................................................................. 3
Chapter 1.        Economic Performance.....................

In [25]:
TARGET_PDF = PDFS[2].name 

print(f"PDF: {TARGET_PDF}")
print("\n" + "="*60)
print("STRATEGY A — first 1200 chars")
print("="*60)
print(results_a[TARGET_PDF][:1200])

print("\n" + "="*60)
print("STRATEGY B — first 1200 chars (flattened blocks)")
print("="*60)
b_text = blocks_to_text(results_b[TARGET_PDF])
print(b_text[:1200])


PDF: FY23_Q4_Consolidated_Financial_Statements.pdf

STRATEGY A — first 1200 chars
 
 
CONDENSED CONSOLIDATED STATEMENTS OF OPERATIONS (Unaudited) 
(In millions, except number of shares, which are reflected in thousands, and per-share amounts) 
 
Three Months Ended 
 
Twelve Months Ended 
 
 
 
 
Net sales: 
 
  
  
  
   Products 
67,184   ! 
70,958   ! 
298,085  ! 
316,199  
 
22,314    
19,188    
85,200   
78,129  
Total net sales (1) 
 
89,498   
90,146   
383,285   
Cost of sales: 
 
  
  
  
   Products 
 
42,586   
46,387   
189,282    
201,471  
 
6,485   
5,664   
24,855    
22,075  
Total cost of sales 
 
49,071   
52,051   
214,137   
223,546 
Gross margin 
 
40,427   
38,095   
169,148   
170,782  
 
 
  
  
  
Operating expenses: 
 
  
  
  
Research and development 
 
7,307    
6,761    
29,915    
26,251  
Selling, general and administrative 
 
6,151    
6,440   
24,932    
25,094 
Total operating expenses 
 
13,458   
13,201   
54,847   
51,345  
 
 
  
  
  
Operating 

In [26]:
def find_near(text: str, keyword: str, window: int = 600) -> str:
    idx = text.lower().find(keyword.lower())
    if idx == -1:
        return f"  [keyword '{keyword}' NOT FOUND]"
    return text[max(0, idx - 30): idx + window]

# Using the financial PDF for table testing
fin_pdf = next((p.name for p in PDFS if "financial" in p.name.lower() or "Q4" in p.name or "annual" in p.name.lower()), PDFS[0].name)

print(f"Table test PDF: {fin_pdf}")
print("\n" + "="*60)
print("STRATEGY A — around 'revenue' or first number-heavy section")
print("="*60)
print(find_near(results_a[fin_pdf], "revenue") or find_near(results_a[fin_pdf], "total"))

print("\n" + "="*60)
print("STRATEGY B — tables extracted")
print("="*60)
tables_b = [b for b in results_b[fin_pdf] if b.type == "table"]
if tables_b:
    for i, t in enumerate(tables_b[:3]):
        print(f"\n--- Table {i+1} (page {t.page}, section: {t.section[:40]}) ---")
        print(t.content[:800])
else:
    print("[No tables detected by pdfplumber on this PDF]")


Table test PDF: Annual-Report-2023.pdf

STRATEGY A — around 'revenue' or first number-heavy section
  [keyword 'revenue' NOT FOUND]

STRATEGY B — tables extracted

--- Table 1 (page 1, section: ) ---
|  |  |  |  |  |  | FRICA |  |  |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
|  |  |  |  |  |  | FRICA |  |  |
| 20
23
Dr. Floribert Ngaruko
Executive Director,
AFRICA GROUP I CONSTITUENCY
ANNUAL
REPORT |  | 20
23
Dr. Floribert Ngaruko
Executive Director,
AFRICA GROUP I CONSTITUENCY
ANNUAL
REPORT |  |  |  |  |  |  |
|  |  |  |  | 3
rt Ngaruko
tor,
I CONSTITUENCY
NUAL
PORT |  |  |  |  |

--- Table 2 (page 7, section: ) ---
|  | ations
D) |
| --- | --- |
| Acronyms & Abbrevi
AfG1 Africa Group 1 Constituency
AfDB African Development Bank
WBL World Bank's Women, Business, and the Law
COVID-19 Coronavirus Disease 2019
Common Framework G20 Common Framework for Debt Treatment
CPF Country Partnership Framework
CRW Crisis Response Window
ED Executive Director
EMDEs Emerging Market and 

In [27]:
print("Headings detected across all PDFs:\n")
for pdf_name, blocks in results_b.items():
    headings = [b for b in blocks if b.type == "heading"]
    print(f"── {pdf_name} ({len(headings)} headings) ──")
    for h in headings[:8]:
        indent = "  " * (h.heading_level - 1) if h.heading_level else ""
        print(f"  {indent}[H{h.heading_level or '?'}] p.{h.page} — {h.content[:70]}")
    if len(headings) > 8:
        print(f"  ... and {len(headings)-8} more")
    print()


Headings detected across all PDFs:

── 1910.10683v4.pdf (15 headings) ──
  [H1] p.1 — Exploring the Limits of Transfer Learning with a Unified
Text-to-Text 
    [H2] p.1 — Colin Raffel ∗
craffel@gmail.com
    [H2] p.1 — Noam Shazeer ∗
noam@google.com
    [H2] p.1 — Adam Roberts ∗
adarob@google.com
    [H2] p.1 — Katherine Lee ∗
katherinelee@google.com
  [H1] p.1 — arXiv:1910.10683v4  [cs.LG]  19 Sep 2023
    [H2] p.1 — Sharan Narang
sharannarang@google.com
    [H2] p.1 — Michael Matena
mmatena@google.com
  ... and 7 more

── Annual-Report-2023.pdf (7 headings) ──
  [H1] p.8 — Selected Development Indicators for
Constituency Countries
    [H2] p.8 — Annual Report 2023
Africa Group 1 Constituency
vi
    [H2] p.9 — Annual Report 2023
Africa Group 1 Constituency
vii
  [H1] p.11 — Foreword
  [H1] p.13 — Executive Summary
  [H1] p.15 — CHAPTER 1
  [H1] p.15 — Economic
Performance

── FY23_Q4_Consolidated_Financial_Statements.pdf (0 headings) ──

── P180107173682d0431bf651fded74199f10.pdf (58

In [28]:
print("Quality gate report (pages with <50 chars = likely scanned/blank):\n")
for pdf in PDFS:
    doc = pymupdf.open(str(pdf))
    pages = []
    for i, page in enumerate(doc):
        if i >= MAX_PAGES:
            break
        pages.append(page.get_text())
    doc.close()
    report = quality_gate(pages)
    low_quality = [r for r in report if "LOW" in r["quality"]]
    status = f"⚠️  {len(low_quality)} low-quality pages" if low_quality else "✅ All pages OK"
    print(f"  {pdf.name}: {status}")
    for r in low_quality:
        print(f"    → Page {r['page']}: {r['chars']} chars")


Quality gate report (pages with <50 chars = likely scanned/blank):

  1910.10683v4.pdf: ✅ All pages OK
  Annual-Report-2023.pdf: ⚠️  3 low-quality pages
    → Page 2: 0 chars
    → Page 4: 0 chars
    → Page 10: 0 chars
  FY23_Q4_Consolidated_Financial_Statements.pdf: ✅ All pages OK
  P180107173682d0431bf651fded74199f10.pdf: ⚠️  4 low-quality pages
    → Page 2: 0 chars
    → Page 3: 39 chars
    → Page 4: 0 chars
    → Page 5: 40 chars
  _10-K-Q4-2023-As-Filed.pdf: ✅ All pages OK


### For Annual-Report-2023.pdf, pages 2,4,10 are empty pages so even our code is marking it as low-quality pages it doesn't matters as it's empty in original pdf
### For P180107173682d0431bf651fded74199f10.pdf also verified manually, it's fine as pages were empty in the original source

## Phase 1 Assessment


## Phase 2: Chunking Strategy Comparison

**Fixed:** Strategy B extraction (from Phase 1)  
**Variable:** Chunking strategy

| Option | Strategy | Idea |
|---|---|---|
| 2A | SlidingWindow (existing) | Token-based windows, blind to structure |
| 2B | SemanticBlock (new) | Chunk at heading/table/code boundaries — never split a table or code block |


In [29]:
from backend.models import Chunk, SourceType

_ENC = tiktoken.encoding_for_model("gpt-4o")

def _token_count(text: str) -> int:
    return len(_ENC.encode(text))

def _sliding_split(text: str, max_tokens: int, overlap_tokens: int) -> list[str]:
    """Apply sliding window only when a paragraph exceeds max_tokens."""
    tokens = _ENC.encode(text)
    if len(tokens) <= max_tokens:
        return [text]
    step = max_tokens - overlap_tokens
    parts = []
    for start in range(0, len(tokens), step):
        window = tokens[start: start + max_tokens]
        if window:
            parts.append(_ENC.decode(window))
    return parts


def semantic_block_chunk(
    blocks: list[ExtractedBlock],
    metadata: dict,
    max_tokens: int = 512,
    overlap_tokens: int = 50,
) -> list[Chunk]:
    """
    Chunking rules:
    - heading → flush current buffer, start new chunk with heading as context
    - table   → always its own chunk (never split)
    - code    → always its own chunk (never split)
    - paragraph → accumulate; flush when buffer exceeds max_tokens
    """
    chunks: list[Chunk] = []
    buffer_texts: list[str] = []
    buffer_tokens = 0
    current_section = metadata.get("source_url", "")

    def _flush(section: str):
        if buffer_texts:
            content = "\n\n".join(buffer_texts)
            chunks.append(Chunk(
                tenant_id=metadata.get("tenant_id", ""),
                source_url=metadata.get("source_url", ""),
                source_type=SourceType(metadata.get("source_type", "pdf")),
                content=content,
                metadata={**metadata, "section": section, "chunk_type": "text"},
            ))

    for block in blocks:
        content = block.content.strip()
        if not content:
            continue

        if block.type == "heading":
            _flush(current_section)
            buffer_texts = [content]
            buffer_tokens = _token_count(content)
            current_section = content

        elif block.type in ("table", "code"):
            _flush(current_section)
            buffer_texts = []
            buffer_tokens = 0
            chunks.append(Chunk(
                tenant_id=metadata.get("tenant_id", ""),
                source_url=metadata.get("source_url", ""),
                source_type=SourceType(metadata.get("source_type", "pdf")),
                content=content,
                metadata={**metadata, "section": current_section, "chunk_type": block.type},
            ))

        else:  # paragraph / caption
            tok = _token_count(content)
            if tok > max_tokens:
                _flush(current_section)
                buffer_texts = []
                buffer_tokens = 0
                for part in _sliding_split(content, max_tokens, overlap_tokens):
                    chunks.append(Chunk(
                        tenant_id=metadata.get("tenant_id", ""),
                        source_url=metadata.get("source_url", ""),
                        source_type=SourceType(metadata.get("source_type", "pdf")),
                        content=part,
                        metadata={**metadata, "section": current_section, "chunk_type": "paragraph"},
                    ))
            elif buffer_tokens + tok > max_tokens:
                _flush(current_section)
                buffer_texts = [content]
                buffer_tokens = tok
            else:
                buffer_texts.append(content)
                buffer_tokens += tok

    _flush(current_section)
    return chunks

print("SemanticBlockChunker defined.")


SemanticBlockChunker defined.


In [30]:
from backend.connectors.chunkers.sliding_window_chunker import SlidingWindowChunker

sw_chunker = SlidingWindowChunker(window_tokens=512, overlap_tokens=50)

chunks_2a = {}  # SlidingWindow on Strategy B text
chunks_2b = {}  # SemanticBlock on Strategy B blocks

for pdf in PDFS:
    name = pdf.name
    blocks = results_b[name]
    flat_text = blocks_to_text(blocks)
    meta = {"tenant_id": "exp", "source_url": name, "source_type": "pdf"}

    chunks_2a[name] = sw_chunker.chunk(flat_text, meta)
    chunks_2b[name] = semantic_block_chunk(blocks, meta)

    print(f"{name}")
    print(f"  2A SlidingWindow : {len(chunks_2a[name])} chunks")
    print(f"  2B SemanticBlock : {len(chunks_2b[name])} chunks")
    types = Counter(c.metadata.get("chunk_type", "?") for c in chunks_2b[name])
    print(f"  SemanticBlock types: {dict(types)}")
    print()


1910.10683v4.pdf
  2A SlidingWindow : 25 chunks
  2B SemanticBlock : 70 chunks
  SemanticBlock types: {'text': 35, 'code': 23, 'paragraph': 11, 'table': 1}

Annual-Report-2023.pdf
  2A SlidingWindow : 10 chunks
  2B SemanticBlock : 21 chunks
  SemanticBlock types: {'table': 5, 'text': 11, 'paragraph': 5}

FY23_Q4_Consolidated_Financial_Statements.pdf
  2A SlidingWindow : 6 chunks
  2B SemanticBlock : 6 chunks
  SemanticBlock types: {'text': 3, 'table': 3}

P180107173682d0431bf651fded74199f10.pdf
  2A SlidingWindow : 29 chunks
  2B SemanticBlock : 76 chunks
  SemanticBlock types: {'text': 76}

_10-K-Q4-2023-As-Filed.pdf
  2A SlidingWindow : 28 chunks
  2B SemanticBlock : 36 chunks
  SemanticBlock types: {'text': 35, 'table': 1}



In [31]:
INSPECT_PDF = PDFS[0].name

print(f"=== Inspecting: {INSPECT_PDF} ===\n")

print("── 2A SlidingWindow — first 3 chunks ──")
for i, chunk in enumerate(chunks_2a[INSPECT_PDF][:3]):
    print(f"\nChunk {i+1} ({_token_count(chunk.content)} tokens):")
    print(chunk.content[:400])
    print("...")

print("\n" + "="*60)
print("── 2B SemanticBlock — first 3 chunks ──")
for i, chunk in enumerate(chunks_2b[INSPECT_PDF][:3]):
    ctype = chunk.metadata.get("chunk_type", "?")
    section = chunk.metadata.get("section", "")[:40]
    print(f"\nChunk {i+1} [{ctype}] section='{section}' ({_token_count(chunk.content)} tokens):")
    print(chunk.content[:400])
    print("...")


=== Inspecting: 1910.10683v4.pdf ===

── 2A SlidingWindow — first 3 chunks ──

Chunk 1 (512 tokens):
# Exploring the Limits of Transfer Learning with a Unified
Text-to-Text Transformer

## Colin Raffel ∗
craffel@gmail.com

## Noam Shazeer ∗
noam@google.com

## Adam Roberts ∗
adarob@google.com

## Katherine Lee ∗
katherinelee@google.com

# arXiv:1910.10683v4  [cs.LG]  19 Sep 2023

## Sharan Narang
sharannarang@google.com

## Michael Matena
mmatena@google.com

## Yanqi Zhou
yanqiz@google.com

## We
...

Chunk 2 (512 tokens):
 model can process text in a way that is amenable to downstream
learning. This can be loosely viewed as developing general-purpose knowledge that allows
the model to “understand” text. This knowledge can range from low-level (e.g. the spelling

```
∗. Equal contribution. A description of each author’s contribution is available in Appendix A . Correspondence
to craffel@gmail.com .
1. https://github
...

Chunk 3 (512 tokens):
 2014 ), pre-training is typically done via

## Eval Set Generation

Generate 8 question-answer pairs per PDF using GPT-4o-mini.  
Questions span: factual lookups, table values, structural questions (what sections exist), and reasoning questions.

Total: ~40 QA pairs across 5 PDFs.


In [35]:
from openai import AsyncOpenAI

client = AsyncOpenAI()

QA_SYSTEM_PROMPT = """You are an expert at creating evaluation datasets for RAG systems.
Given a document excerpt, generate exactly 8 question-answer pairs.

Rules:
- 2 questions about specific facts or numbers (if present)
- 2 questions about table content (if present); skip if no tables
- 2 questions about the document structure or sections
- 2 open-ended reasoning questions

Respond with a JSON object with a single key "qa_pairs" containing an array:
{"qa_pairs": [{"question": "...", "answer": "...", "type": "factual|table|structural|reasoning"}, ...]}

Answers must be answerable from the provided text only. Be specific.
"""

async def generate_qa_pairs(text: str, pdf_name: str, n: int = 8) -> list[dict]:
    excerpt = text[:6000]
    response = await client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.3,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": QA_SYSTEM_PROMPT},
            {"role": "user", "content": f"Document: {pdf_name}\n\n{excerpt}\n\nGenerate {n} QA pairs."}
        ]
    )
    raw = response.choices[0].message.content
    parsed = json.loads(raw)

    # Handle any key GPT decides to use
    if isinstance(parsed, list):
        pairs = parsed
    else:
        # Find the first list value in the response object
        pairs = next(
            (v for v in parsed.values() if isinstance(v, list)),
            []
        )

    for p in pairs:
        p["source_pdf"] = pdf_name
    return pairs[:n]

all_qa_pairs = []
for pdf in PDFS:
    name = pdf.name
    text = blocks_to_text(results_b[name])
    pairs = await generate_qa_pairs(text, name)
    all_qa_pairs.extend(pairs)
    print(f"✓ {name}: {len(pairs)} QA pairs generated")

print(f"\nTotal: {len(all_qa_pairs)} QA pairs")


✓ 1910.10683v4.pdf: 8 QA pairs generated
✓ Annual-Report-2023.pdf: 8 QA pairs generated
✓ FY23_Q4_Consolidated_Financial_Statements.pdf: 8 QA pairs generated
✓ P180107173682d0431bf651fded74199f10.pdf: 8 QA pairs generated
✓ _10-K-Q4-2023-As-Filed.pdf: 8 QA pairs generated

Total: 40 QA pairs


In [36]:
print("Sample QA pairs:\n")
for qa in all_qa_pairs[:6]:
    print(f"[{qa.get('type','?')}] {qa['question']}")
    print(f"  → {qa['answer'][:120]}")
    print()

# Save
qa_path = EVAL_DIR / "pdf_eval_v1.jsonl"
with open(qa_path, "w") as f:
    for qa in all_qa_pairs:
        f.write(json.dumps(qa) + "\n")
print(f"Saved {len(all_qa_pairs)} pairs → {qa_path}")


Sample QA pairs:

[factual] What is the primary focus of the paper?
  → The primary focus of the paper is to explore the landscape of transfer learning techniques for natural language processi

[factual] When was the paper submitted and revised?
  → The paper was submitted on 1/20 and revised on 6/20.

[structural] What is the title of the paper?
  → The title of the paper is 'Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer'.

[structural] Who are the authors of the paper?
  → The authors of the paper are Colin Raffel, Noam Shazeer, Adam Roberts, Katherine Lee, Sharan Narang, Michael Matena, Yan

[reasoning] How does the paper contribute to the field of NLP?
  → The paper contributes to the field of NLP by systematically studying different transfer learning approaches and providin

[reasoning] What is the significance of using unsupervised learning for pre-training in NLP?
  → The significance of using unsupervised learning for pre-training in NLP is t

## RAGAS Scoring

**In-memory pipeline:**
1. Embed all chunks from the strategy under test (OpenAI `text-embedding-3-small`)
2. For each question: embed → cosine similarity → top-5 chunks
3. GPT-4o-mini generates answer from retrieved chunks
4. RAGAS scores: context_precision, context_recall, faithfulness, answer_relevancy


In [ ]:
from openai import AsyncOpenAI
from backend.strategies.embedding.openai_embedding import OpenAIEmbedding

embedder = OpenAIEmbedding()

async def embed_chunks(chunks: list[Chunk]) -> tuple[list[Chunk], np.ndarray]:
    if not chunks:
        return chunks, np.empty((0, 1536), dtype=np.float32)  # empty matrix, safe to skip

    texts = [c.content for c in chunks]
    batch_size = 100
    all_vecs = []
    for i in range(0, len(texts), batch_size):
        vecs = await embedder.embed(texts[i: i + batch_size])
        all_vecs.extend(vecs)
    matrix = np.array(all_vecs, dtype=np.float32)
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    matrix = matrix / np.maximum(norms, 1e-9)
    return chunks, matrix



async def retrieve(query: str, chunks: list[Chunk], matrix: np.ndarray, top_k: int = 5) -> list[Chunk]:
    q_vec = np.array((await embedder.embed([query]))[0], dtype=np.float32)
    q_vec = q_vec / max(np.linalg.norm(q_vec), 1e-9)
    scores = matrix @ q_vec
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [chunks[i] for i in top_idx]


RAG_SYSTEM = "Answer the question using only the provided context. Be specific and concise."

async def generate_answer(question: str, contexts: list[str]) -> str:
    context_text = "\n\n---\n\n".join(contexts)
    resp = await client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {"role": "system", "content": RAG_SYSTEM},
            {"role": "user", "content": f"Context:\n{context_text}\n\nQuestion: {question}"}
        ]
    )
    return resp.choices[0].message.content


async def run_rag_eval(all_chunks: list[Chunk], qa_pairs: list[dict], label: str) -> list[dict]:
    if not all_chunks:
        print(f"  [{label}] SKIPPED — 0 chunks (extraction likely failed)")
        return []
    
    print(f"Embedding {len(all_chunks)} chunks for [{label}]...")
    chunks, matrix = await embed_chunks(all_chunks)

    records = []
    for qa in qa_pairs:
        retrieved = await retrieve(qa["question"], chunks, matrix)
        contexts = [c.content for c in retrieved]
        answer = await generate_answer(qa["question"], contexts)
        records.append({
            "question":     qa["question"],
            "answer":       answer,
            "ground_truth": qa["answer"],
            "contexts":     contexts,
        })

    print(f"  [{label}] {len(records)} answers collected.")
    return records

print("In-memory RAG runner ready.")


In-memory RAG runner ready.


In [38]:
all_chunks_2a = [c for name in chunks_2a for c in chunks_2a[name]]
records_2a = await run_rag_eval(all_chunks_2a, all_qa_pairs, "2A SlidingWindow")


Embedding 98 chunks for [2A SlidingWindow]...
  [2A SlidingWindow] 40 answers collected.


In [39]:
all_chunks_2b = [c for name in chunks_2b for c in chunks_2b[name]]
records_2b = await run_rag_eval(all_chunks_2b, all_qa_pairs, "2B SemanticBlock")


Embedding 209 chunks for [2B SemanticBlock]...
  [2B SemanticBlock] 40 answers collected.


In [40]:
from datasets import Dataset
from ragas import aevaluate
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

judge_llm   = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))
judge_embed = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))
metrics     = [Faithfulness(), AnswerRelevancy(), ContextPrecision(), ContextRecall()]

scores_2a = await aevaluate(Dataset.from_list(records_2a), metrics=metrics,
                             llm=judge_llm, embeddings=judge_embed)
scores_2b = await aevaluate(Dataset.from_list(records_2b), metrics=metrics,
                             llm=judge_llm, embeddings=judge_embed)

def agg(scores) -> dict:
    df = scores.to_pandas()
    cols = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
    return {c: round(df[c].mean(), 4) for c in cols if c in df.columns}

res_2a = agg(scores_2a)
res_2b = agg(scores_2b)

print("Phase 2 Results:")
print(f"{'Metric':<25} {'2A SlidingWindow':>18} {'2B SemanticBlock':>18} {'Delta':>10}")
print("-" * 75)
for m in res_2a:
    delta = res_2b[m] - res_2a[m]
    arrow = "▲" if delta > 0.005 else ("▼" if delta < -0.005 else "~")
    print(f"{m:<25} {res_2a[m]:>18.4f} {res_2b[m]:>18.4f} {arrow}{abs(delta):>8.4f}")

phase2_winner = "2B" if sum(res_2b.values()) > sum(res_2a.values()) else "2A"
print(f"\n✅ Phase 2 winner: {phase2_winner}")


Evaluating:   0%|          | 0/160 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g

Evaluating:   0%|          | 0/160 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g

Phase 2 Results:
Metric                      2A SlidingWindow   2B SemanticBlock      Delta
---------------------------------------------------------------------------
faithfulness                          0.8520             0.7935 ▼  0.0585
answer_relevancy                      0.8955             0.7706 ▼  0.1249
context_precision                     0.6032             0.4878 ▼  0.1154
context_recall                        0.7750             0.6500 ▼  0.1250

✅ Phase 2 winner: 2A


In [41]:
def semantic_block_chunk_v2(
    blocks: list[ExtractedBlock],
    metadata: dict,
    max_tokens: int = 512,
    overlap_tokens: int = 50,
    min_heading_words: int = 5,
) -> list[Chunk]:
    chunks: list[Chunk] = []
    buffer_texts: list[str] = []
    buffer_tokens = 0
    current_section = metadata.get("source_url", "")

    def _flush(section: str):
        if buffer_texts:
            content = "\n\n".join(buffer_texts)
            chunks.append(Chunk(
                tenant_id=metadata.get("tenant_id", ""),
                source_url=metadata.get("source_url", ""),
                source_type=SourceType(metadata.get("source_type", "pdf")),
                content=content,
                metadata={**metadata, "section": section, "chunk_type": "text"},
            ))
            buffer_texts.clear()

    for block in blocks:
        content = block.content.strip()
        if not content:
            continue

        # Treat short "headings" as paragraphs — they're false positives
        is_real_heading = (
            block.type == "heading"
            and len(content.split()) >= min_heading_words
            and not content.endswith("@gmail.com")  # author emails
            and not re.match(r'^[\d\s\.\-]+$', content)  # page numbers
        )

        if is_real_heading:
            _flush(current_section)
            buffer_texts = [content]
            buffer_tokens = _token_count(content)
            current_section = content

        elif block.type in ("table", "code"):
            _flush(current_section)
            chunks.append(Chunk(
                tenant_id=metadata.get("tenant_id", ""),
                source_url=metadata.get("source_url", ""),
                source_type=SourceType(metadata.get("source_type", "pdf")),
                content=content,
                metadata={**metadata, "section": current_section, "chunk_type": block.type},
            ))

        else:  # paragraph, caption, OR short false-heading treated as paragraph
            tok = _token_count(content)
            if tok > max_tokens:
                _flush(current_section)
                for part in _sliding_split(content, max_tokens, overlap_tokens):
                    chunks.append(Chunk(
                        tenant_id=metadata.get("tenant_id", ""),
                        source_url=metadata.get("source_url", ""),
                        source_type=SourceType(metadata.get("source_type", "pdf")),
                        content=part,
                        metadata={**metadata, "section": current_section, "chunk_type": "paragraph"},
                    ))
            elif buffer_tokens + tok > max_tokens:
                _flush(current_section)
                buffer_texts = [content]
                buffer_tokens = tok
            else:
                buffer_texts.append(content)
                buffer_tokens += tok

    _flush(current_section)
    return chunks


# Re-run chunking with v2
chunks_2b_v2 = {}
for pdf in PDFS:
    name = pdf.name
    blocks = results_b[name]
    meta = {"tenant_id": "exp", "source_url": name, "source_type": "pdf"}
    chunks_2b_v2[name] = semantic_block_chunk_v2(blocks, meta)

print("Chunk counts after heading fix:")
print(f"{'PDF':<50} {'2A':>6} {'2B old':>8} {'2B v2':>8}")
print("-" * 75)
for pdf in PDFS:
    name = pdf.name
    print(f"{name:<50} {len(chunks_2a[name]):>6} {len(chunks_2b[name]):>8} {len(chunks_2b_v2[name]):>8}")


Chunk counts after heading fix:
PDF                                                    2A   2B old    2B v2
---------------------------------------------------------------------------
1910.10683v4.pdf                                       25       70       64
Annual-Report-2023.pdf                                 10       21       18
FY23_Q4_Consolidated_Financial_Statements.pdf           6        6        6
P180107173682d0431bf651fded74199f10.pdf                29       76       67
_10-K-Q4-2023-As-Filed.pdf                             28       36       33


In [42]:
all_chunks_2b_v2 = [c for name in chunks_2b_v2 for c in chunks_2b_v2[name]]
records_2b_v2 = await run_rag_eval(all_chunks_2b_v2, all_qa_pairs, "2B SemanticBlock v2")

scores_2b_v2 = await aevaluate(Dataset.from_list(records_2b_v2), metrics=metrics,
                                llm=judge_llm, embeddings=judge_embed)
res_2b_v2 = agg(scores_2b_v2)

print("Phase 2 Updated Results:")
print(f"{'Metric':<25} {'2A SlidingWindow':>18} {'2B old':>12} {'2B v2 (fixed)':>15} {'Delta vs 2A':>13}")
print("-" * 87)
for m in res_2a:
    old_delta  = res_2b[m] - res_2a[m]
    new_delta  = res_2b_v2[m] - res_2a[m]
    arrow = "▲" if new_delta > 0.005 else ("▼" if new_delta < -0.005 else "~")
    print(f"{m:<25} {res_2a[m]:>18.4f} {res_2b[m]:>12.4f} {res_2b_v2[m]:>15.4f} {arrow}{abs(new_delta):>11.4f}")

phase2_winner = "2B_v2" if sum(res_2b_v2.values()) > sum(res_2a.values()) else "2A"
print(f"\n✅ Phase 2 winner: {phase2_winner}")


Embedding 188 chunks for [2B SemanticBlock v2]...
  [2B SemanticBlock v2] 40 answers collected.


Evaluating:   0%|          | 0/160 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g

Phase 2 Updated Results:
Metric                      2A SlidingWindow       2B old   2B v2 (fixed)   Delta vs 2A
---------------------------------------------------------------------------------------
faithfulness                          0.8520       0.7935          0.8931 ▲     0.0411
answer_relevancy                      0.8955       0.7706          0.7919 ▼     0.1036
context_precision                     0.6032       0.4878          0.5394 ▼     0.0638
context_recall                        0.7750       0.6500          0.7250 ▼     0.0500

✅ Phase 2 winner: 2A


## Now running strategy C & D using pymupdf4llm & unstructured io along with 2 chunking strategy i.e. sliding window & recursive character splitting

In [70]:
import importlib
import sys

# Force remove cached modules
for mod_name in list(sys.modules.keys()):
    if "pymupdf" in mod_name.lower() or "fitz" in mod_name.lower():
        del sys.modules[mod_name]

# Verify what Python/venv this kernel is actually using
import sys
print("Python:", sys.executable)
print("Version:", sys.version)

# Re-import fresh
import pymupdf
print("pymupdf:", pymupdf.__version__)

import pymupdf4llm
print("pymupdf4llm: imported OK")


Python: /Users/mdayanarshad/Desktop/Switch Job UAE/kapa-inspired-rag-mcp/venv/bin/python
Version: 3.12.3 (v3.12.3:f6650f9ad7, Apr  9 2024, 08:18:47) [Clang 13.0.0 (clang-1300.0.29.30)]
pymupdf: 1.27.2.3
pymupdf4llm: imported OK


In [71]:
import pymupdf4llm

def extract_strategy_c(pdf_path: Path, max_pages: int = MAX_PAGES) -> str:
    """pymupdf4llm: dedicated PDF→Markdown converter. One call replaces all of Strategy B."""
    # Cap max_pages to actual page count
    doc = pymupdf.open(str(pdf_path))
    actual_pages = min(max_pages, len(doc))
    doc.close()

    md = pymupdf4llm.to_markdown(
        str(pdf_path),
        pages=list(range(actual_pages)),
        show_progress=False,
    )
    pages = md.split("\n-----\n")
    pages = clean_pipeline(pages)
    return "\n\n".join(pages)

results_c = {}
for pdf in PDFS:
    results_c[pdf.name] = extract_strategy_c(pdf)
    print(f"{pdf.name}: {len(results_c[pdf.name].split())} words")

print("\n=== Sample output (Apple Financial) ===")
fin = "FY23_Q4_Consolidated_Financial_Statements.pdf"
print(results_c[fin][:1500])


=== Document parser messages ===
Using Tesseract for OCR processing.

1910.10683v4.pdf: 7759 words
=== Document parser messages ===
                                    Using Tesseract for OCR processing.
OCR on page.number=0/1.
OCR on page.number=2/3.
OCR on page.number=7/8.
OCR on page.number=8/9.
OCR on page.number=10/11.
OCR on page.number=11/12.

Annual-Report-2023.pdf: 2980 words
=== Document parser messages ===
                                                                                                                                                                                                                            Using Tesseract for OCR processing.

FY23_Q4_Consolidated_Financial_Statements.pdf: 426 words
=== Document parser messages ===
                                                                                                                                                                                                                                        

Warning in pixScaleSmooth: ridiculously small scaling factor 0.016393
Image too small to scale!! (1x1 vs min width of 3)
Line cannot be recognized!!


=== Document parser messages ===
                                                                                                                                                                                                                                                                                                                                                                            Using Tesseract for OCR processing.
OCR on page.number=0/1.

_10-K-Q4-2023-As-Filed.pdf: 10167 words

=== Sample output (Apple Financial) ===

## **CONDENSED CONSOLIDATED STATEMENTS OF OPERATIONS (Unaudited)** 

(In millions, except number of shares, which are reflected in thousands, and per-share amounts) 

|Net sales:|**Three Months Ended**|**Three Months Ended**|**Twelve Months Ended**|**Twelve Months Ended**|
|---|---|---|---|---|
||**September 30,**<br>**2023**|**September 24,**<br>**2022**|**September 30,**<br>**2023**|**September 24,**<br>**2022**|
|Products|�<br>67,184|�<br>70,958|�<br>29

In [72]:
from unstructured.partition.pdf import partition_pdf
from unstructured.documents.elements import (
    Title, NarrativeText, Table, ListItem,
    CodeSnippet, Header, Footer, PageBreak,
)

def _table_to_markdown(elem) -> str:
    """Convert unstructured Table element to Markdown."""
    # Try HTML representation first (better structure)
    html = getattr(elem.metadata, "text_as_html", None)
    if html:
        try:
            import re
            # Quick HTML table → Markdown conversion
            rows = re.findall(r'<tr[^>]*>(.*?)</tr>', html, re.DOTALL)
            md_rows = []
            for i, row in enumerate(rows):
                cells = re.findall(r'<t[dh][^>]*>(.*?)</t[dh]>', row, re.DOTALL)
                cells = [re.sub(r'<[^>]+>', '', c).strip() for c in cells]
                md_rows.append("| " + " | ".join(cells) + " |")
                if i == 0:
                    md_rows.append("| " + " | ".join(["---"] * len(cells)) + " |")
            return "\n".join(md_rows)
        except Exception:
            pass
    return elem.text  # fallback to plain text


def extract_strategy_d(pdf_path: Path, max_pages: int = MAX_PAGES) -> list[ExtractedBlock]:
    """unstructured: ML-based layout detection → typed elements."""
    try:
        elements = partition_pdf(
            filename=str(pdf_path),
            strategy="fast",          # "hi_res" needs GPU; "fast" uses pdfminer
            include_page_breaks=True,
        )
    except Exception as e:
        print(f"  [unstructured ERROR] {pdf_path.name}: {e}")
        return []

    blocks: list[ExtractedBlock] = []
    current_section = ""

    for elem in elements:
        page_num = getattr(elem.metadata, "page_number", 1) or 1
        if page_num > max_pages:
            break
        if isinstance(elem, (Header, Footer, PageBreak)):
            continue  # skip noise

        content = elem.text.strip()
        if not content:
            continue

        if isinstance(elem, Title):
            current_section = content
            blocks.append(ExtractedBlock("heading", content, page_num,
                                         section=current_section, heading_level=2))
        elif isinstance(elem, Table):
            md = _table_to_markdown(elem)
            blocks.append(ExtractedBlock("table", md, page_num, section=current_section))
        elif isinstance(elem, CodeSnippet):
            blocks.append(ExtractedBlock("code", content, page_num, section=current_section))
        else:  # NarrativeText, ListItem, etc.
            blocks.append(ExtractedBlock("paragraph", content, page_num, section=current_section))

    return blocks


results_d = {}
for pdf in PDFS:
    blocks = extract_strategy_d(pdf)
    results_d[pdf.name] = blocks
    headings = sum(1 for b in blocks if b.type == "heading")
    tables   = sum(1 for b in blocks if b.type == "table")
    code     = sum(1 for b in blocks if b.type == "code")
    print(f"{pdf.name}: {len(blocks)} blocks | {headings} headings | {tables} tables | {code} code")

print("\n=== Sample: unstructured on Apple Financial ===")
fin_blocks = results_d["FY23_Q4_Consolidated_Financial_Statements.pdf"]
for b in fin_blocks[:6]:
    print(f"[{b.type}] {b.content[:120]}")
    print()


No languages specified, defaulting to English.


1910.10683v4.pdf: 0 blocks | 0 headings | 0 tables | 0 code


No languages specified, defaulting to English.
Ignoring wrong pointing object 7 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)


Annual-Report-2023.pdf: 0 blocks | 0 headings | 0 tables | 0 code


No languages specified, defaulting to English.


FY23_Q4_Consolidated_Financial_Statements.pdf: 0 blocks | 0 headings | 0 tables | 0 code


No languages specified, defaulting to English.
No languages specified, defaulting to English.


P180107173682d0431bf651fded74199f10.pdf: 0 blocks | 0 headings | 0 tables | 0 code
_10-K-Q4-2023-As-Filed.pdf: 0 blocks | 0 headings | 0 tables | 0 code

=== Sample: unstructured on Apple Financial ===


In [73]:
def recursive_chunk(
    text: str,
    metadata: dict,
    max_tokens: int = 512,
    overlap_tokens: int = 50,
) -> list[Chunk]:
    """
    Split on separators in priority order.
    Only moves to the next separator if the chunk is still too large.
    """
    separators = ["\n\n", "\n", ". ", " "]

    def _split_text(text: str, sep_idx: int) -> list[str]:
        if _token_count(text) <= max_tokens:
            return [text]
        if sep_idx >= len(separators):
            return _sliding_split(text, max_tokens, overlap_tokens)

        sep = separators[sep_idx]
        parts = text.split(sep)
        result: list[str] = []
        current = ""

        for part in parts:
            if not part.strip():
                continue
            candidate = (current + sep + part) if current else part
            if _token_count(candidate) <= max_tokens:
                current = candidate
            else:
                if current:
                    result.append(current.strip())
                if _token_count(part) > max_tokens:
                    result.extend(_split_text(part, sep_idx + 1))
                    current = ""
                else:
                    current = part

        if current.strip():
            result.append(current.strip())
        return result

    raw_chunks = _split_text(text, 0)

    # Add token overlap between adjacent chunks
    chunks_out: list[Chunk] = []
    for i, ct in enumerate(raw_chunks):
        if not ct.strip():
            continue
        if i > 0 and overlap_tokens > 0:
            prev_tokens = _ENC.encode(raw_chunks[i - 1])
            overlap_text = _ENC.decode(prev_tokens[-overlap_tokens:])
            ct = overlap_text + " " + ct
        chunks_out.append(Chunk(
            tenant_id=metadata.get("tenant_id", ""),
            source_url=metadata.get("source_url", ""),
            source_type=SourceType(metadata.get("source_type", "pdf")),
            content=ct.strip(),
            metadata={**metadata, "chunk_type": "recursive"},
        ))

    return chunks_out

print("RecursiveChunker defined.")


RecursiveChunker defined.


In [74]:
import re as _re

def sentence_chunk(
    text: str,
    metadata: dict,
    sentences_per_chunk: int = 6,
    overlap_sentences: int = 1,
) -> list[Chunk]:
    """Group N sentences per chunk with sentence-level overlap."""
    # Split on sentence boundaries
    raw = _re.split(r'(?<=[.!?])\s+', text)
    sentences = [s.strip() for s in raw if len(s.strip()) > 15]

    if not sentences:
        return []

    step = max(1, sentences_per_chunk - overlap_sentences)
    chunks_out: list[Chunk] = []

    for i in range(0, len(sentences), step):
        group = sentences[i: i + sentences_per_chunk]
        content = " ".join(group).strip()
        if not content:
            continue
        chunks_out.append(Chunk(
            tenant_id=metadata.get("tenant_id", ""),
            source_url=metadata.get("source_url", ""),
            source_type=SourceType(metadata.get("source_type", "pdf")),
            content=content,
            metadata={**metadata, "chunk_type": "sentence"},
        ))

    return chunks_out

print("SentenceChunker defined.")


SentenceChunker defined.


In [75]:
chunks_c_sw = {}   # Strategy C + SlidingWindow
chunks_c_rc = {}   # Strategy C + Recursive
chunks_c_sn = {}   # Strategy C + Sentence
chunks_d_sw = {}   # Strategy D + SlidingWindow
chunks_d_sb = {}   # Strategy D + SemanticBlock v2

for pdf in PDFS:
    name = pdf.name
    meta = {"tenant_id": "exp", "source_url": name, "source_type": "pdf"}

    # C combinations (text input)
    c_text = results_c[name]
    chunks_c_sw[name] = sw_chunker.chunk(c_text, meta)
    chunks_c_rc[name] = recursive_chunk(c_text, meta)
    chunks_c_sn[name] = sentence_chunk(c_text, meta)

    # D combinations (block input)
    d_blocks = results_d[name]
    d_flat = blocks_to_text(d_blocks)
    chunks_d_sw[name] = sw_chunker.chunk(d_flat, meta)
    chunks_d_sb[name] = semantic_block_chunk_v2(d_blocks, meta)

print(f"{'PDF':<48} {'B+SW':>6} {'C+SW':>6} {'C+RC':>6} {'C+SN':>6} {'D+SW':>6} {'D+SB':>6}")
print("-" * 90)
for pdf in PDFS:
    n = pdf.name
    print(f"{n:<48} "
          f"{len(chunks_2a[n]):>6} "
          f"{len(chunks_c_sw[n]):>6} "
          f"{len(chunks_c_rc[n]):>6} "
          f"{len(chunks_c_sn[n]):>6} "
          f"{len(chunks_d_sw[n]):>6} "
          f"{len(chunks_d_sb[n]):>6}")


PDF                                                B+SW   C+SW   C+RC   C+SN   D+SW   D+SB
------------------------------------------------------------------------------------------
1910.10683v4.pdf                                     25     25     27     67      0      0
Annual-Report-2023.pdf                               10     15     17     27      0      0
FY23_Q4_Consolidated_Financial_Statements.pdf         6      5      9      1      0      0
P180107173682d0431bf651fded74199f10.pdf              29     23     28     37      0      0
_10-K-Q4-2023-As-Filed.pdf                           28     29     32     64      0      0


In [76]:
from unstructured.partition.pdf import partition_pdf

test_pdf = PDFS[2]  # Apple Financial — smallest, fastest to test
print(f"Testing: {test_pdf.name}")

elements = partition_pdf(filename=str(test_pdf), strategy="fast", include_page_breaks=True)
print(f"Total elements returned: {len(elements)}")
print()

for e in elements[:15]:
    etype = type(e).__name__
    text = getattr(e, "text", "NO TEXT ATTR")
    page = getattr(e.metadata, "page_number", "?")
    print(f"  [{etype}] page={page} text='{text[:60]}'")


Ignoring wrong pointing object 7 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)


Testing: FY23_Q4_Consolidated_Financial_Statements.pdf


No languages specified, defaulting to English.


Total elements returned: 0



In [66]:
fin = "FY23_Q4_Consolidated_Financial_Statements.pdf"

print("=== Strategy B (hand-rolled) — table area ===")
b_text = blocks_to_text(results_b[fin])
print(find_near(b_text, "Total net sales")[:600])

print("\n=== Strategy C (pymupdf4llm) — table area ===")
print(find_near(results_c[fin], "Total net sales")[:600])

print("\n=== Strategy D (unstructured) — table area ===")
d_tables = [b for b in results_d[fin] if b.type == "table"]
for i, t in enumerate(d_tables[:2]):
    print(f"\n[Table {i+1}]")
    print(t.content[:400])


=== Strategy B (hand-rolled) — table area ===
8 |  | 85,200 |  | 78,129 |
| Total net sales (1) | 89,498 |  | 90,146 |  | 383,285 |  | 394,328 |
| Cost of sales: |  |  |  |  |  |  |  |
| Products | 42,586 |  | 46,387 |  | 189,282 |  | 201,471 |
| Services | 6,485 |  | 5,664 |  | 24,855 |  | 22,075 |
| Total cost of sales | 49,071 |  | 52,051 |  | 214,137 |  | 223,546 |
| Gross margin | 40,427 |  | 38,095 |  | 169,148 |  | 170,782 |
| Operating expenses: |  |  |  |  |  |  |  |
| Research and development | 7,307 |  | 6,761 |  | 29,915 |  | 26,251 |
| Selling, general and administrative | 6,151 |  | 6,440 |  | 24,932 |  | 25,094 |
| Total op

=== Strategy C (pymupdf4llm) — table area ===
8|�<br>298,085|�<br>316,199|
|Total net sales(1)|89,498|90,146|383,285|394,328|
|Cost of sales:|||||
|Products|42,586|46,387|189,282|201,471|
|Services|6,485|5,664|24,855|22,075|
|Total cost of sales|49,071|52,051|214,137|223,546|
|Gross margin|40,427|38,095|169,148|170,782|
|Operating expenses:|||||
|Res

In [67]:
# Build flat chunk lists
all_c_sw = [c for n in chunks_c_sw for c in chunks_c_sw[n]]
all_c_rc = [c for n in chunks_c_rc for c in chunks_c_rc[n]]
all_c_sn = [c for n in chunks_c_sn for c in chunks_c_sn[n]]
all_d_sw = [c for n in chunks_d_sw for c in chunks_d_sw[n]]
all_d_sb = [c for n in chunks_d_sb for c in chunks_d_sb[n]]

combos = {
    "C + SlidingWindow": all_c_sw,
    "C + Recursive":     all_c_rc,
    "C + Sentence":      all_c_sn,
    "D + SlidingWindow": all_d_sw,
    "D + SemanticBlock": all_d_sb,
}

records_new = {}
for label, chunks in combos.items():
    records_new[label] = await run_rag_eval(chunks, all_qa_pairs, label)
    await asyncio.sleep(2)  # brief pause between runs

print("All answers collected.")


Embedding 97 chunks for [C + SlidingWindow]...
  [C + SlidingWindow] 40 answers collected.
Embedding 113 chunks for [C + Recursive]...
  [C + Recursive] 40 answers collected.
Embedding 196 chunks for [C + Sentence]...
  [C + Sentence] 40 answers collected.
Embedding 0 chunks for [D + SlidingWindow]...


AxisError: axis 1 is out of bounds for array of dimension 1

In [ ]:
scores_new = {}
for label, records in records_new.items():
    print(f"Scoring [{label}]...")
    s = await aevaluate(Dataset.from_list(records), metrics=metrics,
                        llm=judge_llm, embeddings=judge_embed)
    scores_new[label] = agg(s)
    await asyncio.sleep(3)

print("Done scoring.")


In [ ]:
all_results_full = {
    "A: get_text() + SlidingWindow (baseline)": res_2a,
    "B: layout-aware + SlidingWindow (P2 winner)": res_2a,   # same, B+SW = 2A winner
    "B: layout-aware + SemanticBlock v2": res_2b_v2,
    **{f"  {k}": v for k, v in scores_new.items()},
}

metrics_cols = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]

print("=" * 100)
print("FULL EXPERIMENT RESULTS")
print("=" * 100)
print(f"\n{'Strategy':<45} {'Faith':>8} {'AnsRel':>8} {'CtxPrec':>8} {'CtxRec':>8} {'SUM':>8}")
print("-" * 100)

winner_label = ""
winner_sum = 0.0

for label, res in all_results_full.items():
    faith   = res.get("faithfulness", 0)
    rel     = res.get("answer_relevancy", 0)
    prec    = res.get("context_precision", 0)
    rec     = res.get("context_recall", 0)
    total   = faith + rel + prec + rec
    marker  = " ◄ BEST" if total > winner_sum else ""
    if total > winner_sum:
        winner_sum = total
        winner_label = label
    print(f"{label:<45} {faith:>8.4f} {rel:>8.4f} {prec:>8.4f} {rec:>8.4f} {total:>8.4f}{marker}")

print(f"\n{'='*100}")
print(f"WINNER: {winner_label.strip()}")
print(f"This combination gets implemented in PDFConnector.")
